# fiberqc — Wilbrecht lab, DANDI:001340

NAc dLight, rewarded trials. Published data, third lab.

In [5]:
import fiberqc as fqc
from IPython.display import Markdown

rec = fqc.load(
    "datasets/nwb/sub-BSD011_ses-p148_rew.csv",
    events="datasets/nwb/sub-BSD011_ses-p148_rew_events.csv",
)

# the pipeline I actually ran
result = fqc.multiverse(rec, my_pipeline={
    "low_pass": 2.0,
    "bleaching": "double_exp",
    "motion": "OLS",
    "normalization": "dFF",
})

result

/Users/demiregeortac/Documents/GitHub/fiber-tool/fiberqc/io.py:180: UserWarning: sub-BSD011_ses-p148_rew.csv: sampling is not uniform — 2 gap(s), largest 140.1 s (median step 32.7 ms). Filtering assumes uniform sampling, so gaps corrupt the result. Using the longest continuous segment: 5624 s of 6180 s, dropping 9 event(s) outside it.
  signal, control, time_s, ev = _handle_gaps(signal, control, time_s, ev,


MultiverseResult('sub-BSD011_ses-p148_rew.csv': FLIP, 16/16 significant, d=-0.33..0.80, yours d=0.50, t=-5.6..13.7, SIGN FLIPS)

In [6]:
result.where_do_i_stand()

{'my_effect': 0.00310315094412912,
 'my_d': 0.49588086252908936,
 'my_t': 8.531456946353755,
 'my_p': 7.694769459481509e-16,
 'my_significant': True,
 'n_alternatives': 15,
 'agree': 7,
 'opposite_sign': 8,
 'not_significant': 0,
 'd_if_i_had_chosen_otherwise': (-0.3271519522387639, 0.7988540236828992)}

All sixteen analyses of the same recording. The column that decides the result comes first.

In [7]:
result.to_df()

,bleaching,low_pass,motion,normalization,effect,d,t,p,sig,n_events,my_pipeline
0,highpass,2.0,OLS,dFF,-0.004389,-0.327152,-5.628535,4.228907e-08,True,296,False
1,highpass,2.0,OLS,zscore,-0.129789,-0.325332,-5.597220,4.980273e-08,True,296,False
2,highpass,10.0,OLS,dFF,-0.004255,-0.322762,-5.553015,6.266228e-08,True,296,False
3,highpass,10.0,OLS,zscore,-0.119921,-0.320892,-5.520835,7.400259e-08,True,296,False
4,highpass,10.0,robust,dFF,-0.002898,-0.256178,-4.407454,1.466099e-05,True,296,False
5,highpass,10.0,robust,zscore,-0.081188,-0.253951,-4.369133,1.729673e-05,True,296,False
6,highpass,2.0,robust,dFF,-0.002709,-0.243991,-4.197786,3.571842e-05,True,296,False
7,highpass,2.0,robust,zscore,-0.079382,-0.241734,-4.158950,4.196371e-05,True,296,False
8,double_exp,10.0,OLS,dFF,0.002382,0.364994,6.279588,1.212453e-09,True,296,False
9,double_exp,10.0,OLS,zscore,0.004015,0.369161,6.351284,8.059690e-10,True,296,False


Every statistic above is computed in Python. Claude receives them as fixed numbers and writes the interpretation — never the values.

In [8]:
Markdown(result.ask("Hi Claude!, How are you? I found a significant dopamine response to reward. Should I publish this?"))

I'm doing well, thanks for asking! But I'd hold off on publishing this one — the result is not robust, and here's why:

**The core problem: sign flip.** All 16 pipelines are "significant," but they don't agree on the direction of the effect. That's the FLIP verdict, and it's the worst-case outcome for a robustness check — it means "dopamine response to reward" could just as easily be reported as an increase or a decrease depending on arbitrary preprocessing choices.

**What's driving it:** the split lines up almost perfectly with the **bleaching correction method**:
- `bleaching=highpass` → all t negative (-4.16 to -5.63)
- `bleaching=double_exp` → all t positive (+6.28 to +13.74)

Motion correction (OLS vs robust) and low-pass cutoff shift the magnitude somewhat, but they never flip the sign on their own — bleaching correction is the dominant factor here. Normalization (dFF vs zscore) barely matters at all.

**What this means:** your "significant dopamine response" is really an artifact of an analytical choice you probably made without realizing it mattered this much. Highpass filtering and double-exponential detrending are correcting for photobleaching in very different ways, and one of them (or both) is likely distorting the true signal — possibly by removing or reintroducing slow trends that overlap with your reward-related signal.

**Before publishing:**
1. Don't report a direction or effect size from a single pipeline — it's not defensible given this data.
2. Figure out which bleaching-correction assumption is actually appropriate for your recording (check residual trends, bleaching curves, fit diagnostics for both methods on your raw traces).
3. Ideally, validate against an independent ground truth (e.g., known reward-related dopamine dynamics from histology/electrophysiology literature, or an isosbestic control channel) to determine which correction is physiologically valid.
4. If you can't adjudicate confidently, report both outcomes and flag the sensitivity explicitly rather than picking the one that "looks best."

Right now you have a robust *statistical* effect but an unresolved *scientific* one — the numbers don't tell you if dopamine goes up or down with reward in this dataset.